# Línea Base para Predicción de Combates Pokémon

### Importación de Módulos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

c:\Programacion\TFM Predicción Combates Pokémon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Carga de Datos

In [2]:
df_cleaned = pd.read_csv("../data/gen9ou_cleaned_dataset.csv")
df_cleaned.head()

,battle_id,p1_poke1,p1_poke2,p1_poke3,p1_poke4,p1_poke5,p1_poke6,p2_poke1,p2_poke2,p2_poke3,p2_poke4,p2_poke5,p2_poke6,p1_win
0,1636793-gen9ou-2460221181,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,1
1,1636793-gen9ou-2460221181,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,0
2,95801-gen9ou-2513923983,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,1
3,95801-gen9ou-2513923983,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,0
4,1514712-gen9ou-2429668049,Dragonite,Zamazenta,LandorusTherianTherian,Gholdengo,Darkrai,Hatterene,Ninetales,Ceruledge,Venusaur,Hatterene,Great Tusk,Walking Wake,1


## 1. Separación de Conjuntos de Entrenamiento y Prueba

Split según battle_id para evitar data leakage

In [3]:
battle_ids = df_cleaned['battle_id'].unique()

train_val_ids, test_ids = train_test_split(
    battle_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.2, 
    random_state=RANDOM_STATE
)

train_df = df_cleaned[df_cleaned["battle_id"].isin(train_ids)]
val_df   = df_cleaned[df_cleaned["battle_id"].isin(val_ids)]
test_df  = df_cleaned[df_cleaned["battle_id"].isin(test_ids)]

Se guardan en .csv las batallas pertenecientes a cada conjunto para su posterior uso

In [4]:
pd.DataFrame({"battle_id": train_ids}).to_csv("../splits/train_ids.csv", index=False)
pd.DataFrame({"battle_id": val_ids}).to_csv("../splits/val_ids.csv", index=False)
pd.DataFrame({"battle_id": test_ids}).to_csv("../splits/test_ids.csv", index=False)

## 2. Encoding

In [5]:
pokemon_cols = [
    "p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6",
    "p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"
]

all_pokemon = pd.unique(df_cleaned[pokemon_cols].values.ravel())

pokemon_to_idx = {p: i for i, p in enumerate(sorted(all_pokemon))}
n_pokemon = len(pokemon_to_idx)

print("Número de Pokémon:", n_pokemon)

Número de Pokémon: 809


Se codifica el dataset utilizando Multi-hot encoding (manual) con una representación de 1 para cada Pokémon presente en el combate y 0 para los ausentes. Esto se hace para ambos equipos (equipo 1 y equipo 2) y se concatenan las representaciones para formar la matriz de características final.

Además, se utiliza `scipy.sparse.csr_matrix` para convertir la matriz de características a un formato disperso, lo que es eficiente en términos de memoria dado que la mayoría de los valores serán ceros.

In [6]:
def build_sparse_matrix(df, pokemon_to_idx):
    n_rows = len(df)
    n_poke_unique = len(pokemon_to_idx)

    p1_cols = ["p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6"]
    p2_cols = ["p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"]

    p1_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p1_cols], axis=1)
    p2_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p2_cols], axis=1)

    # Offset para el jugador 2
    p2_idx = p2_idx + n_poke_unique

    # Cada fila tendrá 12 índices (6 de p1 y 6 de p2)
    all_indices = np.hstack([p1_idx, p2_idx]) 

    # 'rows' debe repetirse 12 veces por cada fila del DF
    rows = np.repeat(np.arange(n_rows), 12)
    cols = all_indices.flatten()
    data = np.ones(len(rows), dtype=np.int8)

    X = csr_matrix(
        (data, (rows, cols)),
        shape=(n_rows, 2 * n_poke_unique),
        dtype=np.int8
    )

    y = df["p1_win"].to_numpy()
    return X, y

In [7]:
X_train, y_train = build_sparse_matrix(train_df, pokemon_to_idx)
X_val, y_val     = build_sparse_matrix(val_df, pokemon_to_idx)
X_test, y_test   = build_sparse_matrix(test_df, pokemon_to_idx)

print(X_train.shape)

(2255146, 1618)


A continuación, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline de Random Forest, debido a limitaciones de memoria y tiempo de entrenamiento.

In [8]:
RF_SAMPLE_BATTLES = 100_000

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

In [9]:
X_rf_train, y_rf_train = build_sparse_matrix(rf_df, pokemon_to_idx)

## 3. Entrenamiento del Modelo de Baseline

### 3.2 Entrenamiento de Random Forest

Ya que Random Forest no puede manejar matrices dispersas, convertimos a formato denso (aunque esto puede consumir mucha memoria, por lo que se recomienda hacerlo solo con una muestra de los datos).

In [10]:
X_rf_train_dense = X_rf_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()

Se define a continuación el la función objetivo para Optuna, que busca los mejores hiperparámetros para el modelo de Random Forest utilizando la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [11]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_rf_train, y_rf_train)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [12]:
study_name = "optuna_rf_classic_baseline"

study_rf = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective, n_trials=5)


best_params = study_rf.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-04-14 12:46:45,627] Using an existing study with name 'optuna_rf_classic_baseline' instead of creating a new one.
[I 2026-04-14 12:48:08,410] Trial 35 finished with value: 0.5539513312923499 and parameters: {'n_estimators': 526, 'max_depth': 24, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 34 with value: 0.5539627504558094.
[I 2026-04-14 12:48:40,181] Trial 36 finished with value: 0.551320299490089 and parameters: {'n_estimators': 578, 'max_depth': 24, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 34 with value: 0.5539627504558094.
[I 2026-04-14 12:49:59,040] Trial 37 finished with value: 0.5539422389243104 and parameters: {'n_estimators': 518, 'max_depth': 24, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 34 with value: 0.5539627504558094.
[I 2026-04-14 12:50:23,812] Trial 38 finished with value: 0.5461845382589634 and parameters: {'n_estimators': 509, 'max

In [13]:
best_params = study_rf.best_params.copy()

rf_model = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_rf_train_dense, y_rf_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",535
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",24
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",9
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

Tras el entrenamiento con los mejores hiperparámetros, se evalúa el modelo en el conjunto de prueba.

In [14]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

val_proba = rf_model.predict_proba(X_val_dense)[:, 1]


thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [15]:


test_proba = rf_model.predict_proba(X_test_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5373
F1 macro:  0.5360
ROC-AUC:   0.5547
Log Loss:  0.6904

Matriz de confusión:
[[170518 181849]
 [144197 208170]]


Para comparar métricas, se incluye dos baseline naive que predice siempre la clase mayoritaria del conjunto de entrenamiento, y otro que predice aleatoriamente según la distribución de clases del conjunto de entrenamiento.

In [16]:
majority_class = np.bincount(y_train).argmax()
y_pred_naive = np.full_like(y_test, majority_class)
y_pred_random = np.random.randint(0, 2, size=len(y_test))


print("Naive Accuracy:", accuracy_score(y_test, y_pred_naive))
print("Naive F1 macro:", f1_score(y_test, y_pred_naive, average="macro"))
print("Random Accuracy:", accuracy_score(y_test, y_pred_random))
print("Random F1 macro:", f1_score(y_test, y_pred_random, average="macro"))

Naive Accuracy: 0.5
Naive F1 macro: 0.3333333333333333
Random Accuracy: 0.5009464563934761
Random F1 macro: 0.5009463598283824


### 3.3 Entrenamiento de LightGBM

Dado que LightGBM puede manejar matrices dispersas, se entrena directamente con la matriz en formato CSR sin necesidad de convertir a formato denso, lo que es más eficiente en términos de memoria. Por lo tanto, se puede entrenar con un mayor número de muestras sin preocuparse por el consumo de memoria.

In [17]:
# Convertir los conjuntos de entrenamiento, validación y prueba
X_train_float = X_train.astype('float32')
X_val_float = X_val.astype('float32')
X_test_float = X_test.astype('float32')

# Nota: y_train, y_val, y_test suelen estar bien como int o float, 
# pero si quieres asegurar compatibilidad total:
y_train_float = y_train.astype('float32')
y_val_float = y_val.astype('float32')
y_test_float = y_test.astype('float32')

Por compatibilidad con LightGBM, se convierte el tipo de datos de las características a `float32`.

In [18]:
import lightgbm as lgb

def lgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_float,
        y_train_float,
        eval_set=[(X_val_float, y_val_float)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_float)[:,1]

    roc_auc = roc_auc_score(y_val_float, val_proba)

    return roc_auc

Definimos la función objetivo de Optuna para optimizar los hiperparámetros de LightGBM, además de escoger el mejor threshold para convertir las probabilidades de salida en predicciones binarias. Se utiliza la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [19]:
study_name = "optuna_lgb_classic_baseline"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True
)

study.optimize(lgb_objective, n_trials=10)

best_params = study.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-04-14 12:59:35,078] Using an existing study with name 'optuna_lgb_classic_baseline' instead of creating a new one.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.045186 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2075
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1037
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

[I 2026-04-14 13:02:48,551] Trial 30 finished with value: 0.5998403318003418 and parameters: {'n_estimators': 977, 'learning_rate': 0.08585891204891313, 'num_leaves': 229, 'max_depth': 27, 'min_child_samples': 126, 'subsample': 0.8610604229949158, 'colsample_bytree': 0.8973308340563263}. Best is trial 27 with value: 0.6018114169344717.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.745464 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2075
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1037
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

[I 2026-04-14 13:06:09,541] Trial 31 finished with value: 0.6009508597708451 and parameters: {'n_estimators': 1142, 'learning_rate': 0.08554738755990561, 'num_leaves': 230, 'max_depth': 27, 'min_child_samples': 129, 'subsample': 0.8423551661833006, 'colsample_bytree': 0.8899608719033816}. Best is trial 27 with value: 0.6018114169344717.


...

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.793610 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2253
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1126
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

[I 2026-04-14 13:27:13,045] Trial 38 finished with value: 0.5990946010475088 and parameters: {'n_estimators': 1006, 'learning_rate': 0.08023596062294537, 'num_leaves': 208, 'max_depth': 28, 'min_child_samples': 82, 'subsample': 0.9081390296697259, 'colsample_bytree': 0.7269709639108586}. Best is trial 27 with value: 0.6018114169344717.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.678453 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2103
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1051
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[1009]	valid_0's auc: 0.600991


[I 2026-04-14 13:30:15,356] Trial 39 finished with value: 0.6009909676547427 and parameters: {'n_estimators': 1038, 'learning_rate': 0.1286537633550383, 'num_leaves': 222, 'max_depth': 30, 'min_child_samples': 118, 'subsample': 0.8604293028329015, 'colsample_bytree': 0.7937541979719169}. Best is trial 27 with value: 0.6018114169344717.


Best params: {'n_estimators': 1198, 'learning_rate': 0.11165260825267398, 'num_leaves': 235, 'max_depth': 27, 'min_child_samples': 165, 'subsample': 0.8474973707652133, 'colsample_bytree': 0.8847324759331013}
Best score: 0.6018114169344717


Una vez obtenidos los mejores hiperparámetros, se entrena el modelo final de LightGBM con esos parámetros y se evalúa su rendimiento en el conjunto de test.

In [20]:
best_model = lgb.LGBMClassifier(**study.best_params)

best_model.fit(
    X_train_float,
    y_train_float,
    eval_set=[(X_val_float, y_val_float)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model.predict_proba(X_val_float)[:, 1]

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.794778 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2014
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1007
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

Una vez entrenado el modelo, se ajusta el threshold en función de F1-score macro.

In [21]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [22]:
test_proba = best_model.predict_proba(X_test_float)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test_float, test_preds)
f1_macro = f1_score(y_test_float, test_preds, average="macro")
roc_auc = roc_auc_score(y_test_float, test_proba)
ll = log_loss(y_test_float, test_proba)
cm = confusion_matrix(y_test_float, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5705
F1 macro:  0.5704
ROC-AUC:   0.6012
Log Loss:  0.6766

Matriz de confusión:
[[198081 154286]
 [148417 203950]]


## 4. Representación Team Differential

Se basa en la idea de representar cada combate como la diferencia entre los pokémon del equipo 1 y los pokémon del equipo 2. Esto se hace restando la representación de los pokémon del equipo 2 de la representación de los pokémon del equipo 1 de la forma `team_diff_vector = team1_vector - team2_vector`, lo que da como resultado una nueva representación que captura las diferencias entre ambos equipos. 

In [10]:
def build_team_differential(df, pokemon_to_idx):
    n_pokemon = len(pokemon_to_idx)
    
    rows = []
    cols = []
    data = []
    
    y = df['p1_win'].values
    
    for row_idx, row in enumerate(df.itertuples(index=False)):
        # team 1: +1
        for p in [row.p1_poke1,row.p1_poke2,row.p1_poke3,row.p1_poke4,row.p1_poke5,row.p1_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(1)
        
        # team 2: -1
        for p in [row.p2_poke1,row.p2_poke2,row.p2_poke3,row.p2_poke4,row.p2_poke5,row.p2_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(-1)
    
    X = csr_matrix((data, (rows, cols)), shape=(len(df), n_pokemon), dtype=np.int8)
    
    return X, y

Se introduce el experimento de eliminar la duplicación de datos, ya que con esta representación no es necesario invertir los equipos para evitar el sesgo del orden, dado que la diferencia ya captura esa información.

In [11]:
import numpy as np

def remove_battle_duplicates_random(df, seed=42):
    print(f"Original size: {len(df)}")
    
    df_unique = (
        df.sample(frac=1, random_state=seed)  # 🔀 shuffle
          .drop_duplicates(subset="battle_id", keep="first")
          .copy()
    )
    
    print(f"After removing duplicates: {len(df_unique)}")
    print(f"Removed: {len(df) - len(df_unique)} rows")
    
    if "p1_win" in df_unique.columns:
        print("\nClass distribution:")
        print(df_unique["p1_win"].value_counts(normalize=True))
    
    return df_unique



df_nodup = remove_battle_duplicates_random(df_cleaned)

train_df_nodup = df_nodup[df_nodup["battle_id"].isin(train_df["battle_id"])]
val_df_nodup   = df_nodup[df_nodup["battle_id"].isin(val_df["battle_id"])]
test_df_nodup  = df_nodup[df_nodup["battle_id"].isin(test_df["battle_id"])]

Original size: 3523668
After removing duplicates: 1761834
Removed: 1761834 rows

Class distribution:
p1_win
1    0.500319
0    0.499681
Name: proportion, dtype: float64


In [12]:
# Train completo
X_train_diff, y_train = build_team_differential(train_df_nodup, pokemon_to_idx)
X_test_diff, y_test = build_team_differential(test_df_nodup, pokemon_to_idx)
X_val_diff, y_val = build_team_differential(val_df_nodup, pokemon_to_idx)

#hay que meter el conjunto de validación también
#INTERESANTE PROBAR SI CON ESTA REPRESENTACION ES NECESARIA LA DUPLICACION

print("Team Differential shapes:")
print("X_train:", X_train_diff.shape)
print("X_test :", X_test_diff.shape)
print("X_val  :", X_val_diff.shape)

Team Differential shapes:
X_train: (1127573, 809)
X_test : (352367, 809)
X_val  : (281894, 809)


Al igual que antes, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline con esta nueva representación.

In [13]:
RF_SAMPLE_BATTLES = 100_000  # ajustar según memoria

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df_nodup = train_df_nodup[
    train_df_nodup["battle_id"].isin(rf_battle_ids)
]

X_rf_diff, y_rf = build_team_differential(rf_df_nodup, pokemon_to_idx)

# Random Forest necesita matriz densa
X_train_rf_diff_dense = X_rf_diff.toarray()
X_test_rf_diff_dense = X_test_diff.toarray()
X_val_rf_diff_dense = X_val_diff.toarray()

## 5. Entrenamiento de Modelos con Representación Team Differential

In [36]:
## importar lightgbm y rf
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score, log_loss

### 5.1 Entrenamiento de Random Forest con Team Differential

In [19]:


def rf_objective_diff(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_train_rf_diff_dense, y_rf)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val_rf_diff_dense)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [20]:
print("y_train unique:", np.unique(y_rf))
print("y_val unique:", np.unique(y_val))

y_train unique: [0 1]
y_val unique: [0 1]


In [23]:
study_name = "optuna_rf_diff_baseline"

study_rf = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective_diff, n_trials=5)

best_params = study_rf.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-04-14 14:10:07,183] Using an existing study with name 'optuna_rf_diff_baseline' instead of creating a new one.
[I 2026-04-14 14:10:22,302] Trial 2 finished with value: 0.5379152673657297 and parameters: {'n_estimators': 107, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.5379152673657297.
[I 2026-04-14 14:11:21,218] Trial 3 finished with value: 0.5402883850262645 and parameters: {'n_estimators': 583, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.5402883850262645.
[I 2026-04-14 14:12:43,102] Trial 4 finished with value: 0.5427980897603414 and parameters: {'n_estimators': 534, 'max_depth': 13, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 4 with value: 0.5427980897603414.
[I 2026-04-14 14:13:02,105] Trial 5 finished with value: 0.5378913093094673 and parameters: {'n_estimators': 232, 'max_depth': 6,

In [25]:
best_params = study_rf.best_params.copy()

rf_model_diff = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model_diff.fit(X_train_rf_diff_dense, y_rf)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",299
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [26]:
val_proba = rf_model_diff.predict_proba(X_val_rf_diff_dense)[:, 1]

thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.502


In [37]:
test_proba = rf_model_diff.predict_proba(X_test_rf_diff_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5260
F1 macro:  0.5023
ROC-AUC:   0.5444
Log Loss:  0.6922

Matriz de confusión:
[[131137  45247]
 [121778  54205]]


Se almacena tanto el modelo final como el metadata con el mejor threshold encontrado.

In [38]:
model_name = "rf_team_diff_best_baseline"

joblib.dump(rf_model_diff, f"../models/{model_name}.joblib")

print(f"Modelo guardado en: ../models/{model_name}.joblib")

metadata = {
    "model": "RandomForest",
    "representation": "team_differential",
    "dataset": "final",
    "params": best_params,
    "metric": "roc_auc",
    "best_threshold": float(best_thr),
    "threshold_metric": "f1_macro",
    "threshold_range": [0.3, 0.7]
}

with open(f"../models/{model_name}.json", "w") as f:
    json.dump(metadata, f, indent=4)

Modelo guardado en: ../models/rf_team_diff_best_baseline.joblib


### 5.2 Entrenamiento de LightGBM con Team Differential

In [29]:
X_train_diff = X_train_diff.astype('float32')
X_test_diff = X_test_diff.astype('float32')
X_val_diff = X_val_diff.astype('float32')

In [30]:
import lightgbm as lgb

def lgb_objective_diff(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_diff,
        y_train,
        eval_set=[(X_val_diff, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_diff)[:,1]

    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [31]:
study_name = "optuna_lgb_diff_baseline"

study = optuna.create_study(
    study_name=study_name,
    direction="maximize",
    storage="sqlite:///../studies/" + study_name + ".db",
    load_if_exists=True
)

study.optimize(lgb_objective_diff, n_trials=10)

best_params = study.best_params

with open(f"../parameters/{study_name}.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-04-14 14:15:42,530] A new study created in RDB with name: optuna_lgb_diff_baseline


[LightGBM] [Info] Number of positive: 564937, number of negative: 562636
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.388825 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1539
[LightGBM] [Info] Number of data points in the train set: 1127573, number of used features: 513
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501020 -> initscore=0.004081
[LightGBM] [Info] Start training from score 0.004081
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2026-04-14 14:15:54,099] Trial 0 finished with value: 0.566930454833695 and parameters: {'n_estimators': 205, 'learning_rate': 0.09500865865975128, 'num_leaves': 181, 'max_depth': 9, 'min_child_samples': 77, 'subsample': 0.7698314161090023, 'colsample_bytree': 0.9134010403901955}. Best is trial 0 with value: 0.566930454833695.


[LightGBM] [Info] Number of positive: 564937, number of negative: 562636
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.449995 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1539
[LightGBM] [Info] Number of data points in the train set: 1127573, number of used features: 513
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501020 -> initscore=0.004081
[LightGBM] [Info] Start training from score 0.004081
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2026-04-14 14:16:09,756] Trial 1 finished with value: 0.5424012655533388 and parameters: {'n_estimators': 580, 'learning_rate': 0.06876676440215158, 'num_leaves': 200, 'max_depth': 1, 'min_child_samples': 77, 'subsample': 0.9739350316522131, 'colsample_bytree': 0.7025825605487779}. Best is trial 0 with value: 0.566930454833695.


...

[LightGBM] [Info] Number of positive: 564937, number of negative: 562636
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.529191 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1539
[LightGBM] [Info] Number of data points in the train set: 1127573, number of used features: 513
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501020 -> initscore=0.004081
[LightGBM] [Info] Start training from score 0.004081
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

[I 2026-04-14 14:21:51,708] Trial 8 finished with value: 0.5733195165159006 and parameters: {'n_estimators': 283, 'learning_rate': 0.0658737479532119, 'num_leaves': 241, 'max_depth': 15, 'min_child_samples': 74, 'subsample': 0.7716509649147063, 'colsample_bytree': 0.7315506298558843}. Best is trial 4 with value: 0.5796501681520442.


[LightGBM] [Info] Number of positive: 564937, number of negative: 562636
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.404221 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1404
[LightGBM] [Info] Number of data points in the train set: 1127573, number of used features: 468
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501020 -> initscore=0.004081
[LightGBM] [Info] Start training from score 0.004081
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Li

[I 2026-04-14 14:22:16,355] Trial 9 finished with value: 0.566600034518989 and parameters: {'n_estimators': 727, 'learning_rate': 0.13660763440672608, 'num_leaves': 133, 'max_depth': 4, 'min_child_samples': 170, 'subsample': 0.8871107995689307, 'colsample_bytree': 0.9958851603264547}. Best is trial 4 with value: 0.5796501681520442.


Best params: {'n_estimators': 585, 'learning_rate': 0.11114926960048917, 'num_leaves': 111, 'max_depth': 29, 'min_child_samples': 72, 'subsample': 0.7404160668228755, 'colsample_bytree': 0.8906581908483779}
Best score: 0.5796501681520442


In [39]:
best_model_lgbm = lgb.LGBMClassifier(**study.best_params)

best_model_lgbm.fit(
    X_train_diff,
    y_train,
    eval_set=[(X_val_diff, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model_lgbm.predict_proba(X_val_diff)[:, 1]

[LightGBM] [Info] Number of positive: 564937, number of negative: 562636
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.418739 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1560
[LightGBM] [Info] Number of data points in the train set: 1127573, number of used features: 520
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501020 -> initscore=0.004081
[LightGBM] [Info] Start training from score 0.004081
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[584]	valid_0's auc: 0.579236	valid_0's binary_logloss: 0.682643


In [40]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.506


In [41]:
test_proba = best_model_lgbm.predict_proba(X_test_diff)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5550
F1 macro:  0.5545
ROC-AUC:   0.5795
Log Loss:  0.6828

Matriz de confusión:
[[103555  72829]
 [ 83967  92016]]


Se almacena tanto el modelo final como el metadata con el mejor threshold encontrado.

In [42]:
model_name = "lgb_team_diff_best_baseline"

joblib.dump(best_model_lgbm, f"../models/{model_name}.joblib")

print(f"Modelo guardado en: ../models/{model_name}.joblib")

metadata = {
    "model": "LightGBM",
    "representation": "team_differential",
    "dataset": "final",
    "params": best_params,
    "metric": "roc_auc",
    "best_threshold": float(best_thr),
    "threshold_metric": "f1_macro",
    "threshold_range": [0.3, 0.7]
}

with open(f"../models/{model_name}.json", "w") as f:
    json.dump(metadata, f, indent=4)

Modelo guardado en: ../models/lgb_team_diff_best_baseline.joblib


## 6. Conclusiones

Ya que las obtenidas con la representación de Team Differential son iguales a las obtenidas con la representación original, pero genera menos columnas, se concluye que la representación de Team Differential es más eficiente y no pierde información relevante para la predicción, por lo que en adelante se utilizará esta representación.



In [43]:
df_cleaned_actual_metagame = pd.read_csv("../data/gen9ou_actual_metagame_03-26_cleaned_dataset.csv")


df_cleaned_no_duplicates_actual_metagame = remove_battle_duplicates_random(df_cleaned_actual_metagame)

df_nodup.to_csv("../data/gen9ou_cleaned_no_duplicates.csv", index=False)
df_cleaned_no_duplicates_actual_metagame.to_csv("../data/gen9ou_cleaned_no_duplicates_actual_metagame.csv", index=False)

print(f"Tamaño original: {len(df_cleaned)}")
print(f"Tamaño sin duplicados: {len(df_nodup)}")
print(f"Tamaño sin duplicados (actual metagame): {len(df_cleaned_no_duplicates_actual_metagame)}")

Original size: 2430046
After removing duplicates: 1215023
Removed: 1215023 rows

Class distribution:
p1_win
0    0.500203
1    0.499797
Name: proportion, dtype: float64
Tamaño original: 3523668
Tamaño sin duplicados: 1761834
Tamaño sin duplicados (actual metagame): 1215023
